The traveling salesman problem (TSP) from https://colab.research.google.com/github/Gurobi/modeling-examples/blob/master/traveling_salesman/tsp_gcl.ipynb#scrollTo=q7F3MgLRAUOP



In [13]:
#!pip install folium
import gurobipy as gp
from gurobipy import GRB
import json
import math
from itertools import combinations
import folium

In [14]:
def get_data(local_file, url):
    try:
        capitals_json = json.load(open(local_file))
    except:
      import urllib.request      
      data = urllib.request.urlopen(url).read()
      capitals_json = json.loads(data)

    capitals = []
    coordinates = {}
    for state in capitals_json:
        if state not in ['AK', 'HI']:
          capital = capitals_json[state]['capital']
          capitals.append(capital)
          coordinates[capital] = (float(capitals_json[state]['lat']), float(capitals_json[state]['long']))
    return capitals, coordinates

In [15]:
# Compute pairwise distance matrix
def distance(city1, city2):
    c1 = coordinates[city1]
    c2 = coordinates[city2]
    diff = (c1[0]-c2[0], c1[1]-c2[1])
    return math.sqrt(diff[0]*diff[0]+diff[1]*diff[1])

In [16]:
# tested with Python 3.7 & Gurobi 9.0.0
def make_model(dist):
    m = gp.Model()

    # Variables: is city 'i' adjacent to city 'j' on the tour?
    vars = m.addVars(dist.keys(), obj=dist, vtype=GRB.BINARY, name='x')

    # Symmetric direction: Copy the object
    for i, j in vars.keys():
        vars[j, i] = vars[i, j]  # edge in opposite direction

    # Constraints: two edges incident to each city
    cons = m.addConstrs(vars.sum(c, '*') == 2 for c in capitals)
    return m, vars

In [17]:
def subtourelim(model, where):
    if where == GRB.Callback.MIPSOL:
        # make a list of edges selected in the solution
        vals = model.cbGetSolution(model._vars)
        selected = gp.tuplelist((i, j) for i, j in model._vars.keys()
                             if vals[i, j] > 0.5)
        # find the shortest cycle in the selected edge list
        tour = subtour(selected)
        if len(tour) < len(capitals):
            # add subtour elimination constr. for every pair of cities in subtour
            model.cbLazy(gp.quicksum(model._vars[i, j] for i, j in combinations(tour, 2))
                         <= len(tour)-1)

# Given a tuplelist of edges, find the shortest subtour

def subtour(edges):
    unvisited = capitals[:]
    cycle = capitals[:] # Dummy - guaranteed to be replaced
    while unvisited:  # true if list is non-empty
        thiscycle = []
        neighbors = unvisited
        while neighbors:
            current = neighbors[0]
            thiscycle.append(current)
            unvisited.remove(current)
            neighbors = [j for i, j in edges.select(current, '*')
                         if j in unvisited]
        if len(thiscycle) <= len(cycle):
            cycle = thiscycle # New shortest subtour
    return cycle

def solve_model(model, vars):
    model._vars = vars
    model.Params.lazyConstraints = 1
    model.optimize(subtourelim)
    return model

In [18]:
def validate_model(model):
    vals = model.getAttr('x', vars)
    selected = gp.tuplelist((i, j) for i, j in vals.keys() if vals[i, j] > 0.5)

    tour = subtour(selected)
    assert len(tour) == len(capitals)
    return tour

In [19]:
def draw_map(tour, coordinates):
    map = folium.Map(location=[40,-95], zoom_start = 4)

    points = []
    for city in tour:
      points.append(coordinates[city])
    points.append(points[0])

    folium.PolyLine(points).add_to(map)

    return map

In [20]:
# get the initial data
local_file = 'capitals.json'
url = 'https://raw.githubusercontent.com/Gurobi/modeling-examples/master/traveling_salesman/capitals.json'
capitals, coordinates = get_data(local_file, url)

# calculate distances
dist = {(c1, c2): distance(c1, c2) for c1, c2 in combinations(capitals, 2)}

# create the model
model, vars = make_model(dist)

#solve the model
model = solve_model(model, vars)

# validate the model and extract the "tour"
tour = validate_model(model)

# draw the map of the tour
map = draw_map(tour, coordinates)

map

Set parameter LazyConstraints to value 1
Gurobi Optimizer version 10.0.2 build v10.0.2rc0 (win64)

CPU model: 11th Gen Intel(R) Core(TM) i7-1195G7 @ 2.90GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 48 rows, 1128 columns and 2256 nonzeros
Model fingerprint: 0x63641a38
Variable types: 0 continuous, 1128 integer (1128 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [6e-01, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 2e+00]
Presolve time: 0.01s
Presolved: 48 rows, 1128 columns, 2256 nonzeros
Variable types: 0 continuous, 1128 integer (1128 binary)

Root relaxation: objective 1.611858e+02, 72 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0  161.18576    0   12          -  161.1